# Export the 5-model cascade to CoreML

Ports the Android `.onnx` export pipeline to CoreML `.mlpackage`s for the iOS app. Reads the same
`.pt` checkpoints described in the Android `Model weights` README and produces:

| checkpoint | -> | mlpackage | arch | notes |
|---|---|---|---|---|
| `roi_detector.pt` | -> | `ROIDetector.mlpackage` | YOLO11n, 1 class (`CS`), imgsz 640 | Arch A, conf 0.25 / iou 0.5, take **top-1** box in Swift |
| `pa_detector.pt` | -> | `PADetector.mlpackage` | YOLO11m, 1 class (`pa`), imgsz **1024** | Arch A, conf 0.10 / iou 0.5, run on ROI crop, cap at **100** boxes in Swift |
| `resnet_binary.pt` | -> | `ResNetBinary.mlpackage` | ResNet-18, 2-class (`A`, `NA-OF`), 224x224 | Arch A, stop cascade if P(A) < 0.4 |
| `resnet_subtype.pt` | -> | `ResNetSubtype.mlpackage` | ResNet-18, 3-class (`A-AM`, `A-C`, `A-CRO`), 224x224 | Arch A, only when P(A) >= 0.4; fall back to `A` if top conf < 0.25 |
| `yolo_nano_detector.pt` | -> | `YOLONanoDetector.mlpackage` | YOLO11n, 7 classes, imgsz 640 | Arch B (standalone), conf 0.25 / iou 0.5, uncapped |

Both YOLO exports use Ultralytics' native `format="coreml"` with `nms=True`, which bakes NMS
into the graph as a pipeline stage -- this is what lets Vision hand back
`VNRecognizedObjectObservation`s directly with no manual decode/NMS code needed on the Swift side
(same requirement called out in this folder's existing `README.md`).

**Not exported here / not baked into the model:** the top-1 selection for `roi_detector`, the
100-box cap for `pa_detector`, and the P(A) cascade-routing logic between the two ResNets. Those
are runtime behaviors -- they belong in `InferenceManager.swift`, not in the `.mlpackage` files.

In [ ]:
%pip install --upgrade ultralytics coremltools torch torchvision

In [ ]:
import shutil
from pathlib import Path

import coremltools as ct
import torch
import torch.nn as nn
import torchvision.models as tv_models
from ultralytics import YOLO

# Where the source .pt checkpoints live (same layout as the Android weights folder).
SOURCE_DIR = Path("weights")

# Absolute path to the Xcode target's MLModels group -- NOT Path(".") relative to cwd, since the
# Jupyter kernel's working directory depends on how the notebook was launched (e.g. VS Code
# defaults it to the workspace root, not this notebook's own folder) and silently writes the
# .mlpackages to the wrong place.
OUTPUT_DIR = Path("/Users/michaelrolfe/Desktop/apps/ResnetYoloTester/ResnetYoloTester/MLModels")
print(f"OUTPUT_DIR = {OUTPUT_DIR}")

## YOLO detectors (`roi_detector`, `pa_detector`, `yolo_nano_detector`)

`conf`/`iou` get baked into the exported NMS pipeline at export time -- CoreML has no way to
override them at inference time the way ONNX Runtime session options can, so these values must be
correct here.

In [ ]:
def export_yolo_to_coreml(pt_path: Path, imgsz: int, conf: float, iou: float, out_name: str,
                           output_dir: Path = OUTPUT_DIR) -> Path:
    model = YOLO(str(pt_path))
    exported = Path(model.export(
        format="coreml",
        imgsz=imgsz,
        nms=True,       # bake NMS into the graph -> Vision returns VNRecognizedObjectObservation
        conf=conf,
        iou=iou,
        half=False,      # fp32 graph; the ANE still executes it fp16 internally
        int8=False,
    ))

    output_dir.mkdir(parents=True, exist_ok=True)
    dest = output_dir / f"{out_name}.mlpackage"
    if dest.exists():
        shutil.rmtree(dest)
    shutil.move(str(exported), str(dest))

    print(f"{pt_path.name} -> {dest}  (imgsz={imgsz}, conf={conf}, iou={iou})")
    return dest


YOLO_DETECTORS = [
    dict(pt="roi_detector.pt",       imgsz=640,  conf=0.25, iou=0.50, out_name="ROIDetector"),
    dict(pt="pa_detector.pt",        imgsz=1024, conf=0.10, iou=0.50, out_name="PADetector"),
    dict(pt="yolo_nano_detector.pt", imgsz=640,  conf=0.25, iou=0.50, out_name="YOLONanoDetector"),
]

for cfg in YOLO_DETECTORS:
    export_yolo_to_coreml(
        pt_path=SOURCE_DIR / cfg["pt"],
        imgsz=cfg["imgsz"],
        conf=cfg["conf"],
        iou=cfg["iou"],
        out_name=cfg["out_name"],
    )

## ResNet-18 classifiers (`resnet_binary`, `resnet_subtype`)

CoreML's `ImageType` preprocessing only supports a single scalar `scale` (plus a per-channel
`bias`) -- it can't express a per-channel `std` divide. So instead of relying on CoreML's built-in
image preprocessing for the full ImageNet normalization, we bake `(x - mean) / std` into the traced
graph itself and only use `scale=1/255` at the CoreML boundary to turn the raw 0-255 pixel buffer
into a 0-1 float tensor.

`classifier_config=ct.ClassifierConfig(class_labels)` is what makes Vision hand back
`VNClassificationObservation`s with human-readable identifiers (matching
`InferenceManager.classify`'s `request.results as? [VNClassificationObservation]`), instead of a
raw probability `MLMultiArray`.

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]


class NormalizedResNet(nn.Module):
    """resnet18 with ImageNet mean/std normalization baked into the forward pass."""

    def __init__(self, num_classes: int):
        super().__init__()
        self.backbone = tv_models.resnet18(weights=None)
        self.backbone.fc = nn.Linear(self.backbone.fc.in_features, num_classes)
        self.register_buffer("mean", torch.tensor(IMAGENET_MEAN).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor(IMAGENET_STD).view(1, 3, 1, 1))

    def forward(self, x):
        # x arrives 0-1 float RGB; CoreML's ImageType(scale=1/255) already did the /255 step.
        x = (x - self.mean) / self.std
        return self.backbone(x)


def export_resnet_to_coreml(pt_path: Path, out_name: str, class_labels: list[str],
                             output_dir: Path = OUTPUT_DIR) -> Path:
    wrapped = NormalizedResNet(num_classes=len(class_labels))
    state_dict = torch.load(pt_path, map_location="cpu")
    wrapped.backbone.load_state_dict(state_dict)
    wrapped.eval()

    example_input = torch.rand(1, 3, 224, 224)
    traced = torch.jit.trace(wrapped, example_input)

    mlmodel = ct.convert(
        traced,
        source="pytorch",
        inputs=[ct.ImageType(name="input", shape=example_input.shape, scale=1 / 255.0, bias=[0, 0, 0])],
        classifier_config=ct.ClassifierConfig(class_labels),
        convert_to="mlprogram",
        minimum_deployment_target=ct.target.iOS16,
        compute_units=ct.ComputeUnit.ALL,
    )

    output_dir.mkdir(parents=True, exist_ok=True)
    dest = output_dir / f"{out_name}.mlpackage"
    mlmodel.save(str(dest))

    print(f"{pt_path.name} -> {dest}  ({len(class_labels)}-class: {class_labels})")
    return dest


RESNET_CLASSIFIERS = [
    dict(pt="resnet_binary.pt", out_name="ResNetBinary", class_labels=["A", "NA-OF"]),
    dict(pt="resnet_subtype.pt", out_name="ResNetSubtype", class_labels=["A-AM", "A-C", "A-CRO"]),
]

for cfg in RESNET_CLASSIFIERS:
    export_resnet_to_coreml(
        pt_path=SOURCE_DIR / cfg["pt"],
        out_name=cfg["out_name"],
        class_labels=cfg["class_labels"],
    )

## Sanity check

Reload every exported `.mlpackage` and print its input/output spec -- confirm the YOLO models show
an image input plus `confidence`/`coordinates` (or NMS) outputs, and the ResNets show a
`classLabel` + probability-dictionary output rather than a raw tensor.

In [ ]:
for mlpackage in sorted(OUTPUT_DIR.glob("*.mlpackage")):
    print(f"\n=== {mlpackage.name} ===")
    print(ct.models.MLModel(str(mlpackage)))

## Swift-side follow-up (not done by this notebook)

This produces **5** models where `InferenceManager.swift` currently only knows about 2
(`YOLOv11nDetector`, `ResNetClassifier`). Once these `.mlpackage`s are dropped in and added to the
Xcode target, `InferenceManager` needs to be rewritten to:

- Load all 5 models (`ROIDetector`, `PADetector`, `ResNetBinary`, `ResNetSubtype`,
  `YOLONanoDetector`).
- Run Architecture A as the 4-stage cascade: `ROIDetector` (top-1 box) -> crop -> `PADetector`
  (on the crop, cap 100 boxes) -> `ResNetBinary` -> if `P(A) >= 0.4`, `ResNetSubtype` (falling back
  to the generic `A` label if its top confidence is < 0.25).
- Run Architecture B as the standalone `YOLONanoDetector` call (unchanged from today, just a
  renamed model file).

Say the word and I'll make those Swift changes + update `MLModels/README.md` to match.